In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load images
img_src = cv2.imread('189_0.bmp')
ref = cv2.imread('190_0.bmp')

# Separate input's and target's image color channel
imgr, imgg, imgb = cv2.split(img_src)
imgr2, imgg2, imgb2 = cv2.split(ref)

# Compute input's image histogram
Hnimgr = cv2.calcHist([imgr], [0], None, [256], [0, 256]) / imgr.size
Hnimgg = cv2.calcHist([imgg], [0], None, [256], [0, 256]) / imgg.size
Hnimgb = cv2.calcHist([imgb], [0], None, [256], [0, 256]) / imgb.size

# Compute target's image histogram and histogram specification
Hnimgr2 = cv2.calcHist([imgr2], [0], None, [256], [0, 256]) / imgr2.size
Hnimgg2 = cv2.calcHist([imgg2], [0], None, [256], [0, 256]) / imgg2.size
Hnimgb2 = cv2.calcHist([imgb2], [0], None, [256], [0, 256]) / imgb2.size

# Histogram matching
def hist_match(source, template):
    oldshape = source.shape
    source = source.ravel()
    template = template.ravel()
    s_values, bin_idx, s_counts = np.unique(source, return_inverse=True, return_counts=True)
    t_values, t_counts = np.unique(template, return_counts=True)
    
    s_quantiles = np.cumsum(s_counts).astype(np.float64)
    s_quantiles /= s_quantiles[-1]
    
    t_quantiles = np.cumsum(t_counts).astype(np.float64)
    t_quantiles /= t_quantiles[-1]
    
    interp_t_values = np.interp(s_quantiles, t_quantiles, t_values)
    
    return interp_t_values[bin_idx].reshape(oldshape)

outr = hist_match(imgr, imgr2)
outg = hist_match(imgg, imgg2)
outb = hist_match(imgb, imgb2)

histsp = cv2.merge((outr, outg, outb))

# Display images and histograms
fig, axs = plt.subplots(2, 3, figsize=(15, 10))
axs[0, 0].imshow(cv2.cvtColor(ref, cv2.COLOR_BGR2RGB))
axs[0, 0].set_title('Target Image')
axs[0, 1].imshow(cv2.cvtColor(img_src, cv2.COLOR_BGR2RGB))
axs[0, 1].set_title('Input Image')
axs[0, 2].imshow(cv2.cvtColor(histsp, cv2.COLOR_BGR2RGB))
axs[0, 2].set_title('Result Image')

axs[1, 0].plot(Hnimgr, color='r')
axs[1, 0].set_title('Red Input')
axs[1, 1].plot(Hnimgg, color='g')
axs[1, 1].set_title('Green Input')
axs[1, 2].plot(Hnimgb, color='b')
axs[1, 2].set_title('Blue Input')

fig, axs = plt.subplots(2, 3, figsize=(15, 10))
axs[0, 0].plot(Hnimgr2, color='r')
axs[0, 0].set_title('Red Ref')
axs[0, 1].plot(Hnimgg2, color='g')
axs[0, 1].set_title('Green Ref')
axs[0, 2].plot(Hnimgb2, color='b')
axs[0, 2].set_title('Blue Ref')

axs[1, 0].hist(outr.ravel(), bins=256, color='r', alpha=0.7)
axs[1, 0].set_title('Red Result')
axs[1, 1].hist(outg.ravel(), bins=256, color='g', alpha=0.7)
axs[1, 1].set_title('Green Result')
axs[1, 2].hist(outb.ravel(), bins=256, color='b', alpha=0.7)
axs[1, 2].set_title('Blue Result')

for ax in axs.flat:
    ax.label_outer()

plt.show()

ValueError: not enough values to unpack (expected 3, got 0)